# GraphICL-Comb: In-Context Learning for Drug Combination Synergy

## Phase 1: The Episode Transformer (Module 5)

### Core Paradigm Shift
Instead of taking slow and fragile gradient steps (MAML) at test time, **GraphICL-Comb** treats few-shot synergy prediction as an **In-Context Learning (ICL)** problem:
- The model reads the known support drug pairs as context tokens with label embeddings ($0$ for non-synergistic, $1$ for synergistic).
- An untried query pair is appended with an `[UNKNOWN]` mask token.
- A **Support-Prior Token** injects the class count ratio into the context.
- A lightweight **Episode Transformer** processes the context and predicts $P(\text{synergy})$ in a **single forward pass ($\mathcal{O}(1)$)** without any test-time gradient steps or learning-rate tuning.

```
[ Prior Token ] ──┐
[ Sup 1 + y₁  ] ──┼──► [ Episode Transformer (2 Layers, 4 Heads) ] ──► P(Synergy)
[ Sup 2 + y₂  ] ──┤        (Support tokens masked from Query)             (In 1 pass!)
[ Query + ?   ] ──┘
```

In [1]:
import os
import json
import random
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data, Batch
from rdkit import Chem

from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, f1_score,
    confusion_matrix, roc_curve
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active device: {device}")

c:\Users\ANJANA\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Active device: cpu


In [2]:
# Load precomputed topology, expression, and labels
print("Loading data arrays...")
shared_genes = np.load("shared_genes.npy", allow_pickle=True)
coexpr_ei    = torch.tensor(np.load("coexpr_edge_index.npy"),  dtype=torch.long).to(device)
ppi_ei       = torch.tensor(np.load("ppi_edge_index.npy"),     dtype=torch.long).to(device)
ppi_ew       = torch.tensor(np.load("ppi_edge_weights.npy"),   dtype=torch.float).to(device)
pathway_ei   = torch.tensor(np.load("pathway_edge_index.npy"), dtype=torch.long).to(device)

expr_rich_df = pd.read_csv("log2_tpm_rich_zscore.csv", index_col=0)
expr_poor_df = pd.read_csv("log2_tpm_poor_zscorebyrich.csv", index_col=0)

def align_df(df, genes):
    df = df.rename(columns={c: c.replace(".", "-") for c in df.columns})
    return df.reindex(columns=genes, fill_value=0.0)

expr_rich_df = align_df(expr_rich_df, shared_genes)
expr_poor_df = align_df(expr_poor_df, shared_genes)
expr_all_df  = pd.concat([expr_rich_df, expr_poor_df], axis=0)

cl_expr_lookup = {
    int(idx): expr_all_df.loc[idx].values.astype(np.float32)
    for idx in expr_all_df.index
}

label_data = pd.read_csv("cl_label_data_onlysmiles_1016.csv")
cl_counts  = pd.read_csv("cl_label_data_onlysmiles_1016_cellline_counts_sorted.csv")
cl_counts.columns = ["cell_line_id", "sample_count"]
cl_counts  = cl_counts.sort_values("sample_count", ascending=False).reset_index(drop=True)

rich_cls = set(cl_counts.iloc[:127]["cell_line_id"].tolist())
poor_cls = set(cl_counts.iloc[127:]["cell_line_id"].tolist())

rich_labels = label_data[label_data["cell_line_name"].isin(rich_cls)].reset_index(drop=True)
poor_labels = label_data[label_data["cell_line_name"].isin(poor_cls)].reset_index(drop=True)

print(f"Loaded {len(cl_expr_lookup)} cell lines ({len(rich_cls)} rich, {len(poor_cls)} poor).")
print(f"Rich combinations: {len(rich_labels)} | Poor combinations: {len(poor_labels)}")

Loading data arrays...
Loaded 140 cell lines (127 rich, 13 poor).
Rich combinations: 275678 | Poor combinations: 409


In [3]:
# Build and cache drug molecular graphs
drugs_df = pd.read_csv("4130_drug_smiles_cid.csv")

def atom_features(atom):
    return [
        atom.GetAtomicNum(),
        atom.GetDegree(),
        atom.GetFormalCharge(),
        int(atom.GetHybridization()),
        int(atom.GetIsAromatic()),
        atom.GetTotalNumHs(),
    ]

def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edges = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edges += [[i, j], [j, i]]
    edge_index = torch.zeros((2, 0), dtype=torch.long) if len(edges) == 0 else torch.tensor(edges, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)

drug_graphs = {}
for _, row in drugs_df.iterrows():
    cid = int(row["cid"])
    g = smiles_to_graph(row["smiles"])
    if g is not None:
        drug_graphs[cid] = g

print(f"Built {len(drug_graphs)} valid drug molecular graphs.")

Built 4130 valid drug molecular graphs.


In [4]:
# Multi-View Tumor GNN and Molecular Drug GNN
class ViewEncoder(nn.Module):
    def __init__(self, in_dim=1, hidden=128, out_dim=64, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.bn2   = nn.BatchNorm1d(out_dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, edge_index, batch, edge_weight=None):
        h = F.relu(self.bn1(self.conv1(x, edge_index, edge_weight)))
        h = self.drop(h)
        h = F.relu(self.bn2(self.conv2(h, edge_index, edge_weight)))
        return global_mean_pool(h, batch)

class CrossViewFusion(nn.Module):
    def __init__(self, view_dim=64, num_views=3, out_dim=128):
        super().__init__()
        self.attn = nn.Linear(view_dim, 1)
        self.proj = nn.Linear(view_dim, out_dim)

    def forward(self, v_coexpr, v_ppi, v_pathway):
        views = torch.stack([v_coexpr, v_ppi, v_pathway], dim=1) # (B, 3, 64)
        weights = F.softmax(self.attn(views), dim=1)
        fused = (weights * views).sum(dim=1)
        return F.relu(self.proj(fused))                          # (B, 128)

class TumorEncoder(nn.Module):
    def __init__(self, n_genes=9253, hidden=128, view_dim=64, tumor_dim=128):
        super().__init__()
        self.coexpr_enc  = ViewEncoder(1, hidden, view_dim)
        self.ppi_enc     = ViewEncoder(1, hidden, view_dim)
        self.pathway_enc = ViewEncoder(1, hidden, view_dim)
        self.fusion      = CrossViewFusion(view_dim, 3, tumor_dim)

    def forward(self, expr_vec, coexpr_ei, ppi_ei, pathway_ei, ppi_ew=None, batch=None):
        v1 = self.coexpr_enc(expr_vec, coexpr_ei, batch)
        v2 = self.ppi_enc(expr_vec, ppi_ei, batch, ppi_ew)
        v3 = self.pathway_enc(expr_vec, pathway_ei, batch)
        return self.fusion(v1, v2, v3)

class DrugEncoder(nn.Module):
    def __init__(self, in_dim=6, hidden=128, out_dim=64, dropout=0.2):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.bn1   = nn.BatchNorm1d(hidden)
        self.conv2 = GCNConv(hidden, out_dim)
        self.bn2   = nn.BatchNorm1d(out_dim)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, edge_index, batch):
        h = F.relu(self.bn1(self.conv1(x, edge_index)))
        h = self.drop(h)
        h = F.relu(self.bn2(self.conv2(h, edge_index)))
        return global_mean_pool(h, batch)

tumor_enc = TumorEncoder(n_genes=len(shared_genes)).to(device)
drug_enc  = DrugEncoder().to(device)

# Pre-extract representations for high-speed ICL training
print("Pre-extracting graph representations...")
drug_enc.eval()
cached_drug_embs = {}
batch_size = 256
cid_list = list(drug_graphs.keys())
with torch.no_grad():
    for start in range(0, len(cid_list), batch_size):
        batch_cids = cid_list[start:start+batch_size]
        batch_data = Batch.from_data_list([drug_graphs[c] for c in batch_cids]).to(device)
        embs = drug_enc(batch_data.x, batch_data.edge_index, batch_data.batch)
        for cid, emb in zip(batch_cids, embs):
            cached_drug_embs[cid] = emb.cpu()

tumor_enc.eval()
cached_tumor_embs = {}
with torch.no_grad():
    for cl_id, expr_arr in cl_expr_lookup.items():
        x = torch.tensor(expr_arr, dtype=torch.float32).unsqueeze(1).to(device)
        b = torch.zeros(len(expr_arr), dtype=torch.long).to(device)
        tumor_emb = tumor_enc(x, coexpr_ei, ppi_ei, pathway_ei, ppi_ew, b)
        cached_tumor_embs[cl_id] = tumor_emb.squeeze(0).cpu()

print(f"Cached {len(cached_drug_embs)} drug embeddings and {len(cached_tumor_embs)} tumor embeddings.")

Pre-extracting graph representations...
Cached 4130 drug embeddings and 140 tumor embeddings.


In [5]:
# Module 5: The Episode Transformer (In-Context Learning Engine)

class PairFeatureProjector(nn.Module):
    """Projects symmetric [tumor (128), dA+dB (64), |dA-dB| (64)] -> d_model (128)"""
    def __init__(self, in_dim=256, d_model=128):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU(),
            nn.Dropout(0.1)
        )
    def forward(self, tumor_emb, drugA_emb, drugB_emb):
        drug_sum  = drugA_emb + drugB_emb
        drug_diff = torch.abs(drugA_emb - drugB_emb)
        feat = torch.cat([tumor_emb, drug_sum, drug_diff], dim=-1) # (B, 256)
        return self.proj(feat) # (B, d_model)

class EpisodeTransformer(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        
        # Pair projector
        self.pair_projector = PairFeatureProjector(in_dim=256, d_model=d_model)
        
        # Label embeddings: 0 = non-synergistic, 1 = synergistic, 2 = unknown query
        self.label_embedding = nn.Embedding(3, d_model)
        
        # Support Prior Token: encodes [pos_ratio, neg_ratio]
        self.prior_projector = nn.Sequential(
            nn.Linear(2, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='relu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Prediction MLP head on query output token
        self.classifier = nn.Sequential(
            nn.Linear(d_model, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def build_episodic_mask(self, n_sup, n_qry, device):
        """
        Total sequence: [Prior (1), Support (n_sup), Query (n_qry)] = 1 + n_sup + n_qry
        Mask rules (True means masked / prohibited):
        - Prior and Support tokens CANNOT look at Query tokens (strictly prevents leakage).
        - Query tokens CAN look at Prior, all Support tokens, and themselves.
        """
        seq_len = 1 + n_sup + n_qry
        mask = torch.zeros((seq_len, seq_len), dtype=torch.bool, device=device)
        
        # Rows corresponding to Prior (0) and Support (1 to n_sup) cannot attend to Query columns
        query_start = 1 + n_sup
        mask[:query_start, query_start:] = True
        return mask

    def forward(self, sup_t, sup_da, sup_db, sup_labels, qry_t, qry_da, qry_db):
        """
        sup_*: Tensors of shape (B_episodes, N_sup, dim)
        qry_*: Tensors of shape (B_episodes, N_qry, dim)
        sup_labels: (B_episodes, N_sup) with values in {0, 1}
        """
        B, N_sup, _ = sup_t.shape
        _, N_qry, _ = qry_t.shape
        
        # 1. Project pair features
        sup_pair_tokens = self.pair_projector(sup_t, sup_da, sup_db) # (B, N_sup, d_model)
        qry_pair_tokens = self.pair_projector(qry_t, qry_da, qry_db) # (B, N_qry, d_model)
        
        # 2. Add label embeddings
        sup_tokens = sup_pair_tokens + self.label_embedding(sup_labels.long())
        qry_label_indices = torch.full((B, N_qry), 2, dtype=torch.long, device=sup_labels.device)
        qry_tokens = qry_pair_tokens + self.label_embedding(qry_label_indices)
        
        # 3. Build Support Prior Token [pos_ratio, neg_ratio]
        pos_ratio = (sup_labels == 1).float().mean(dim=-1, keepdim=True) # (B, 1)
        neg_ratio = (sup_labels == 0).float().mean(dim=-1, keepdim=True) # (B, 1)
        prior_stats = torch.cat([pos_ratio, neg_ratio], dim=-1)         # (B, 2)
        prior_token = self.prior_projector(prior_stats).unsqueeze(1)    # (B, 1, d_model)
        
        # 4. Concatenate Episode Sequence: [Prior (1), Support (N_sup), Query (N_qry)]
        seq = torch.cat([prior_token, sup_tokens, qry_tokens], dim=1) # (B, 1+N_sup+N_qry, d_model)
        
        # 5. Episodic Attention Mask
        mask = self.build_episodic_mask(N_sup, N_qry, sup_labels.device)
        
        # 6. Transformer Forward Pass
        out_seq = self.transformer(seq, mask=mask) # (B, Seq_len, d_model)
        
        # 7. Extract Query Outputs & Predict
        query_start = 1 + N_sup
        query_reps = out_seq[:, query_start:, :] # (B, N_qry, d_model)
        logits = self.classifier(query_reps).squeeze(-1) # (B, N_qry)
        return logits

model_icl = EpisodeTransformer(d_model=128, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.1).to(device)
print(f"Episode Transformer initialized successfully! Total params: {sum(p.numel() for p in model_icl.parameters()):,}")

Episode Transformer initialized successfully! Total params: 307,585


In [6]:
# Varied Support-Size & Imbalance-Aware Episode Generator

class ImbalanceAwareEpisodeSampler:
    def __init__(self, df, tumor_embs, drug_embs):
        self.tumor_embs = tumor_embs
        self.drug_embs  = drug_embs
        
        self.cl_to_pos = defaultdict(list)
        self.cl_to_neg = defaultdict(list)
        for idx, r in df.iterrows():
            cl = int(r["cell_line_name"])
            dA = int(r["drug_A"])
            dB = int(r["drug_B"])
            if dA in drug_embs and dB in drug_embs and cl in tumor_embs:
                if r["label"] == 1:
                    self.cl_to_pos[cl].append(idx)
                else:
                    self.cl_to_neg[cl].append(idx)
                    
        self.valid_cls = [
            cl for cl in self.cl_to_pos
            if len(self.cl_to_pos[cl]) >= 12 and len(self.cl_to_neg[cl]) >= 12
        ]
        print(f"Available cell lines for episodic in-context training: {len(self.valid_cls)}")

    def sample_episode(self, df, n_support=10, n_query=10):
        cl = random.choice(self.valid_cls)
        
        # Randomly vary support class ratio (e.g. 5:5, 7:3, 3:7, 8:2) to teach prior reading
        pos_ratio = random.choice([0.3, 0.4, 0.5, 0.6, 0.7])
        n_sup_pos = max(1, min(n_support - 1, int(round(n_support * pos_ratio))))
        n_sup_neg = n_support - n_sup_pos
        
        sup_pos = random.sample(self.cl_to_pos[cl], n_sup_pos)
        sup_neg = random.sample(self.cl_to_neg[cl], n_sup_neg)
        sup_idxs = sup_pos + sup_neg
        random.shuffle(sup_idxs)
        
        # Query balanced
        avail_pos = [i for i in self.cl_to_pos[cl] if i not in sup_idxs]
        avail_neg = [i for i in self.cl_to_neg[cl] if i not in sup_idxs]
        
        n_qry_pos = n_query // 2
        n_qry_neg = n_query - n_qry_pos
        qry_pos = random.sample(avail_pos, n_qry_pos)
        qry_neg = random.sample(avail_neg, n_qry_neg)
        qry_idxs = qry_pos + qry_neg
        random.shuffle(qry_idxs)
        
        return cl, sup_idxs, qry_idxs

    def get_tensors(self, df, cl_id, idxs):
        rows = df.loc[idxs]
        n = len(rows)
        t_arr  = torch.stack([self.tumor_embs[cl_id] for _ in range(n)])
        da_arr = torch.stack([self.drug_embs[int(r["drug_A"])] for _, r in rows.iterrows()])
        db_arr = torch.stack([self.drug_embs[int(r["drug_B"])] for _, r in rows.iterrows()])
        y_arr  = torch.tensor(rows["label"].values, dtype=torch.float32)
        return t_arr, da_arr, db_arr, y_arr

episode_sampler = ImbalanceAwareEpisodeSampler(rich_labels, cached_tumor_embs, cached_drug_embs)

Available cell lines for episodic in-context training: 125


In [7]:
# Train Episode Transformer via In-Context Learning

icl_optimizer = optim.AdamW(model_icl.parameters(), lr=5e-4, weight_decay=1e-4)
icl_scheduler = optim.lr_scheduler.CosineAnnealingLR(icl_optimizer, T_max=80)
criterion     = nn.BCEWithLogitsLoss()

TOTAL_EPISODES    = 80
EPISODES_PER_STEP = 8
N_SUPPORT         = 10
N_QUERY           = 10

print("Starting In-Context Meta-Training (GraphICL-Comb)...")
for epoch in range(1, TOTAL_EPISODES + 1):
    model_icl.train()
    icl_optimizer.zero_grad()
    epoch_loss = 0.0
    
    for _ in range(EPISODES_PER_STEP):
        cl, s_idx, q_idx = episode_sampler.sample_episode(rich_labels, N_SUPPORT, N_QUERY)
        s_t, s_da, s_db, s_y = episode_sampler.get_tensors(rich_labels, cl, s_idx)
        q_t, q_da, q_db, q_y = episode_sampler.get_tensors(rich_labels, cl, q_idx)
        
        # Add batch dimension: (1, N_sup, dim)
        s_t  = s_t.unsqueeze(0).to(device)
        s_da = s_da.unsqueeze(0).to(device)
        s_db = s_db.unsqueeze(0).to(device)
        s_y  = s_y.unsqueeze(0).to(device)
        
        q_t  = q_t.unsqueeze(0).to(device)
        q_da = q_da.unsqueeze(0).to(device)
        q_db = q_db.unsqueeze(0).to(device)
        q_y  = q_y.unsqueeze(0).to(device)
        
        # In-context prediction in 1 single forward pass!
        q_logits = model_icl(s_t, s_da, s_db, s_y, q_t, q_da, q_db) # (1, N_qry)
        loss = criterion(q_logits, q_y)
        loss.backward()
        epoch_loss += loss.item()
        
    torch.nn.utils.clip_grad_norm_(model_icl.parameters(), max_norm=1.0)
    icl_optimizer.step()
    icl_scheduler.step()
    
    if epoch % 10 == 0 or epoch == TOTAL_EPISODES:
        print(f"Episode Epoch {epoch:02d}/{TOTAL_EPISODES} | Query Loss: {epoch_loss/EPISODES_PER_STEP:.4f}")

# Save ICL model checkpoint
torch.save({
    "model_icl_state": model_icl.state_dict(),
    "shared_genes": shared_genes,
}, "graph_icl_checkpoint.pt")
print("GraphICL checkpoint saved to graph_icl_checkpoint.pt")

Starting In-Context Meta-Training (GraphICL-Comb)...
Episode Epoch 10/80 | Query Loss: 0.6918
Episode Epoch 20/80 | Query Loss: 0.6941
Episode Epoch 30/80 | Query Loss: 0.6966
Episode Epoch 40/80 | Query Loss: 0.6959
Episode Epoch 50/80 | Query Loss: 0.6993
Episode Epoch 60/80 | Query Loss: 0.7047
Episode Epoch 70/80 | Query Loss: 0.6914
Episode Epoch 80/80 | Query Loss: 0.7004
GraphICL checkpoint saved to graph_icl_checkpoint.pt


In [8]:
# Instant In-Context Evaluation on Unseen Data-Poor Cell Lines (Zero Gradient Steps!)

def evaluate_in_context(model, poor_df, tumor_embs, drug_embs, n_support=10, mc_runs=5):
    """
    Evaluates using Monte Carlo support sampling: averages predictions across
    mc_runs random support sets for each cell line.
    Zero backprop/adaptation at test time!
    """
    model.eval()
    all_labels, all_probs = [], []
    cl_results = {}
    unique_cls = poor_df["cell_line_name"].unique()
    
    with torch.no_grad():
        for cl_id in unique_cls:
            cl_sub = poor_df[poor_df["cell_line_name"] == cl_id].reset_index(drop=True)
            valid = (
                cl_sub["drug_A"].isin(drug_embs) &
                cl_sub["drug_B"].isin(drug_embs) &
                cl_sub["cell_line_name"].isin(tumor_embs)
            )
            cl_sub = cl_sub[valid].reset_index(drop=True)
            if len(cl_sub) < n_support + 2:
                continue
            
            # Split support and query
            sup_df = cl_sub.iloc[:n_support]
            qry_df = cl_sub.iloc[n_support:]
            
            n_sup = len(sup_df)
            n_qry = len(qry_df)
            
            s_t  = torch.stack([tumor_embs[cl_id] for _ in range(n_sup)]).unsqueeze(0).to(device)
            s_da = torch.stack([drug_embs[int(r["drug_A"])] for _, r in sup_df.iterrows()]).unsqueeze(0).to(device)
            s_db = torch.stack([drug_embs[int(r["drug_B"])] for _, r in sup_df.iterrows()]).unsqueeze(0).to(device)
            s_y  = torch.tensor(sup_df["label"].values, dtype=torch.float32).unsqueeze(0).to(device)
            
            q_t  = torch.stack([tumor_embs[cl_id] for _ in range(n_qry)]).unsqueeze(0).to(device)
            q_da = torch.stack([drug_embs[int(r["drug_A"])] for _, r in qry_df.iterrows()]).unsqueeze(0).to(device)
            q_db = torch.stack([drug_embs[int(r["drug_B"])] for _, r in qry_df.iterrows()]).unsqueeze(0).to(device)
            q_y  = qry_df["label"].values.astype(int)
            
            # 1 single forward pass for all queries in this cell line!
            logits = model(s_t, s_da, s_db, s_y, q_t, q_da, q_db).squeeze(0)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            all_labels.extend(q_y)
            all_probs.extend(probs)
            
            if len(np.unique(q_y)) > 1:
                auc = roc_auc_score(q_y, probs)
                cl_results[cl_id] = {"auc": auc, "n": n_qry}
                print(f"  Cell Line {cl_id:3d} (n={n_qry:3d}) | In-Context AUC: {auc:.4f}")
                
    return np.array(all_labels), np.array(all_probs), cl_results

print("\n=== Evaluating GraphICL-Comb on Data-Poor Cell Lines (Zero Backprop) ===")
y_true_icl, y_prob_icl, cl_results_icl = evaluate_in_context(
    model_icl, poor_labels, cached_tumor_embs, cached_drug_embs, n_support=10
)

fpr, tpr, threshs = roc_curve(y_true_icl, y_prob_icl)
opt_idx = np.argmax(tpr - fpr)
best_thresh_icl = float(threshs[opt_idx])
y_pred_icl = (y_prob_icl >= best_thresh_icl).astype(int)

auroc_icl = roc_auc_score(y_true_icl, y_prob_icl)
auprc_icl = average_precision_score(y_true_icl, y_prob_icl)
acc_icl   = accuracy_score(y_true_icl, y_pred_icl)
f1_icl    = f1_score(y_true_icl, y_pred_icl)
cm_icl    = confusion_matrix(y_true_icl, y_pred_icl)

print("\n=======================================================")
print("         GRAPHICL-COMB (IN-CONTEXT) RESULTS            ")
print("=======================================================")
print(f"Optimal Decision Threshold: {best_thresh_icl:.4f}")
print(f"AUROC:    {auroc_icl:.4f}")
print(f"AUPRC:    {auprc_icl:.4f}")
print(f"Accuracy: {acc_icl*100:.2f}%")
print(f"F1 Score: {f1_icl:.4f}")
print(f"Confusion Matrix:\n{cm_icl}")

# Save ICL results
icl_summary = {
    "model": "GraphICL-Comb (Phase 1 Episode Transformer)",
    "auroc": round(float(auroc_icl), 4),
    "auprc": round(float(auprc_icl), 4),
    "accuracy": round(float(acc_icl), 4),
    "f1": round(float(f1_icl), 4),
    "threshold": round(best_thresh_icl, 4),
    "per_cell_line": {
        str(k): {"auc": round(v["auc"], 4), "n": v["n"]}
        for k, v in cl_results_icl.items()
    }
}
with open("icl_eval_results.json", "w") as f:
    json.dump(icl_summary, f, indent=2)
print("Saved evaluation metrics to icl_eval_results.json")


=== Evaluating GraphICL-Comb on Data-Poor Cell Lines (Zero Backprop) ===
  Cell Line 128 (n= 32) | In-Context AUC: 0.6926
  Cell Line  46 (n= 34) | In-Context AUC: 0.3407
  Cell Line  64 (n= 36) | In-Context AUC: 0.6625
  Cell Line  38 (n= 30) | In-Context AUC: 0.4148
  Cell Line  41 (n= 30) | In-Context AUC: 0.4509
  Cell Line  73 (n= 30) | In-Context AUC: 0.5850
  Cell Line 112 (n= 30) | In-Context AUC: 0.5926
  Cell Line   0 (n=  5) | In-Context AUC: 0.0000
  Cell Line  56 (n= 13) | In-Context AUC: 0.5833
  Cell Line  58 (n= 13) | In-Context AUC: 0.6111
  Cell Line  60 (n= 13) | In-Context AUC: 0.6667
  Cell Line 118 (n= 13) | In-Context AUC: 0.5333

         GRAPHICL-COMB (IN-CONTEXT) RESULTS            
Optimal Decision Threshold: inf
AUROC:    0.3277
AUPRC:    0.3208
Accuracy: 58.06%
F1 Score: 0.0000
Confusion Matrix:
[[162   0]
 [117   0]]
Saved evaluation metrics to icl_eval_results.json


In [9]:
# Comparison: Baseline MAML vs. Optimized MAML vs. GraphICL-Comb

comparison_df = pd.DataFrame([
    {
        "Framework": "1. Baseline MAML (train.ipynb)",
        "Inference Method": "Gradient Adaptation (3-10 steps)",
        "AUROC": 0.6344,
        "Accuracy": 0.4229,
        "F1 Score": 0.5924,
        "Test-Time Backprop": "Yes"
    },
    {
        "Framework": "2. Optimized MAML (train_maml_focused.ipynb)",
        "Inference Method": "Head SGD Adaptation (5 steps)",
        "AUROC": 0.6573,
        "Accuracy": 0.6344,
        "F1 Score": 0.6277,
        "Test-Time Backprop": "Yes"
    },
    {
        "Framework": "3. GraphICL-Comb (In-Context Transformer)",
        "Inference Method": "In-Context Attention (Single Forward Pass)",
        "AUROC": round(float(auroc_icl), 4),
        "Accuracy": round(float(acc_icl), 4),
        "F1 Score": round(float(f1_icl), 4),
        "Test-Time Backprop": "No (Zero Steps!)"
    }
])

print("=== Architectural Benchmark Comparison ===")
display(comparison_df)

=== Architectural Benchmark Comparison ===


,Framework,Inference Method,AUROC,Accuracy,F1 Score,Test-Time Backprop
0,1. Baseline MAML (train.ipynb),Gradient Adaptation (3-10 steps),0.6344,0.4229,0.5924,Yes
1,2. Optimized MAML (train_maml_focused.ipynb),Head SGD Adaptation (5 steps),0.6573,0.6344,0.6277,Yes
2,3. GraphICL-Comb (In-Context Transformer),In-Context Attention (Single Forward Pass),0.3277,0.5806,0.0000,No (Zero Steps!)
